# Full reaction-space prediction

Refit the original five CatBoost models and predict the external reaction-space CSVs. The original model settings are unchanged. Descriptor-retained rows are aligned explicitly. The external full descriptor map is intentional; a compact map may omit reported reactants. Running this notebook trains models and updates external prediction CSVs.

Original notebooks and all saved outputs are preserved in `notebooks/archive/discovery_before_split_20260928/`. This reorganized notebook has not been executed.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "Code" / "model_train.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter within the CycloAddGenerator repository.")
for folder in (REPO_ROOT, REPO_ROOT / "notebooks",
               REPO_ROOT / "notebooks/04_discovery", REPO_ROOT / "notebooks/05_mechanism"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.chdir(REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
from _workflow_paths import *
from catboost import CatBoostRegressor
from Code import model_train


In [ ]:
def load_aligned(csv_path):
    frame = pd.read_csv(require_file(csv_path))
    features, retained = model_train.read_df(frame, str(require_file(DESCRIPTOR_MAP)))
    return frame.iloc[np.asarray(retained, dtype=int)].reset_index(drop=True), features

data_G, X_G = load_aligned(DFT_DIR / "Rxn.csv")
data_Ea, X_Ea = load_aligned(DFT_DIR / "Active_Energy.csv")
params = dict(task_type="CPU", iterations=10000, learning_rate=0.01, depth=6, verbose=0)
model0 = CatBoostRegressor(**params)
model0.fit(X_G, data_G["deltaG"])
X_Ea_G = np.column_stack([X_Ea, model0.predict(X_Ea)])
target_models = {}
for target in ["Diene_Distort", "Ene_Distort", "Interaction", "deltaGa(Solvent)"]:
    estimator = CatBoostRegressor(**params)
    estimator.fit(X_Ea_G, data_Ea[target])
    target_models[target] = estimator


In [ ]:
prediction_csvs = sorted(PRED_DIR.glob("Diene_*.csv"))
if not prediction_csvs:
    raise FileNotFoundError(f"No Diene_*.csv reaction-space files found in {PRED_DIR}")
for csv_path in tqdm(prediction_csvs):
    frame = pd.read_csv(csv_path)
    features, retained = model_train.read_df(frame, str(DESCRIPTOR_MAP))
    row_ids = np.asarray(retained, dtype=int)
    if not len(row_ids):
        raise ValueError(f"No descriptor-covered reactions in {csv_path}")
    predicted_delta_g = model0.predict(features)
    augmented = np.column_stack([features, predicted_delta_g])
    np.save(csv_path.with_name(csv_path.stem + "_des.npy"), np.column_stack([augmented, row_ids]))
    frame["deltaG_p"] = np.nan
    frame.loc[row_ids, "deltaG_p"] = predicted_delta_g
    for target, estimator in target_models.items():
        frame[target + "_p"] = np.nan
        frame.loc[row_ids, target + "_p"] = estimator.predict(augmented)
    # Preserve the legacy alias. This is not a separate gas-phase prediction.
    frame["deltaGa_p"] = frame["deltaGa(Solvent)_p"]
    frame.to_csv(csv_path, index=False)
